In [1]:
import os
import random
import string
import math
import sys
import pandas as pd

import hail as hl
from ukb_utils import hail_init
from ukb_utils import samples
from ukb_utils import variants
from ko_utils import io
from ko_utils import ko

In [2]:
def get_tid(length=5):
    r"""method for getting random ID string for alleles"""
    return ''.join(random.choices(string.ascii_lowercase,
                                  k=length))

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Initializing Hail on SGE with 1 CPU(s).


2022-12-22 10:00:57 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2022-12-22 10:00:57 WARN  SparkConf:69 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.1.3
SparkUI available at http://compa007.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.102-817f6fb3468f
LOGGING: writing to logs/hail/hail_test_export.log


In [6]:
#mt = hl.read_matrix_table("data/unphased/imputed/common_new/ukb_imp_200k_common_chr21.mt/")
mt = hl.read_matrix_table("data/unphased/imputed/common_append_missing/ukb_imp_200k_common_append_missing_chr21.mt//")

In [7]:
mt.count()

(0, 176587)

In [13]:
ko_path = "data/knockouts/alt/pp90/combined/ukb_eur_wes_200k_chr21_pLoF_damaging_missense.mt"
imp_path = "data/unphased/imputed/common_new/ukb_imp_200k_common_chr21.mt"
calls_path = "data/unphased/calls/liftover/ukb_liftover_calls_500k_chr21.mt"
samples_extract = "data/phenotypes/samples/kos_not_in_imp.txt"


In [14]:
ko = io.import_table(ko_path, "mt", calc_info = False)
imp = hl.read_matrix_table(imp_path)
calls = hl.read_matrix_table(calls_path)

In [15]:
ht = hl.import_table(samples_extract, no_header = True, key = 'f0')
calls = calls.filter_cols(hl.is_defined(ht[calls.col_key]))

cols = calls.count_cols()
print("the cols: " + str(cols))

# set all variants to missing
calls = calls.annotate_rows(idrow=1)
calls = calls.filter_rows(calls.idrow==2)

2022-12-22 11:20:27 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)


the cols: 321


In [18]:


# select rows
calls = calls.select_rows(*[calls.rsid])
imp = imp.select_rows(*[imp.rsid])

# export imputed
mt = imp.union_cols(calls, row_join_type='outer')
#mt = tables.order_cols(mt, ko)
mt.count()

(102250, 176587)

In [ ]:
# annotate rows
calls = calls.annotate_rows(
            varid = hl.delimit(
                [hl.str(calls.locus.contig),
                 hl.str(calls.locus.position),
                 calls.alleles[0],
                 calls.alleles[1]],
                ':')
            )

In [15]:
input_path = "data/mt/prefilter/pp90/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp90.maf0_005.mt/"
input_type = "mt"
out_prefix = "test"
csqs_category = ["synonymous"]
subset_gene = None
chunk_idx = 2
genes_per_chunk = 200

In [37]:
# import phased/unphased data
mt = io.import_table(input_path, input_type, calc_info=False)
mt = mt.repartition(32)

In [38]:
assert int(genes_per_chunk) > 0
assert int(chunk_idx) > 0
genes =  mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.collect()
genes = sorted(list(set(genes)))

In [32]:
genes = sorted(list(set(genes)))

In [33]:
chunk_idx=1
idx_start = (int(chunk_idx)-1) * int(genes_per_chunk)
idx_end = (int(chunk_idx)) * int(genes_per_chunk)
subset_gene = genes[idx_start:idx_end]
print("gene subset")
print(subset_gene)

gene subset
['ENSG00000141956', 'ENSG00000141959', 'ENSG00000142149', 'ENSG00000142156', 'ENSG00000142166', 'ENSG00000142168', 'ENSG00000142173', 'ENSG00000142182', 'ENSG00000142185', 'ENSG00000142188', 'ENSG00000142192', 'ENSG00000142197', 'ENSG00000142207', 'ENSG00000154639', 'ENSG00000154640', 'ENSG00000154642', 'ENSG00000154645', 'ENSG00000154646', 'ENSG00000154654', 'ENSG00000154719', 'ENSG00000154721', 'ENSG00000154723', 'ENSG00000154727', 'ENSG00000154734', 'ENSG00000154736', 'ENSG00000155304', 'ENSG00000155307', 'ENSG00000155313', 'ENSG00000156239', 'ENSG00000156253', 'ENSG00000156256', 'ENSG00000156261', 'ENSG00000156265', 'ENSG00000156273', 'ENSG00000156282', 'ENSG00000156284', 'ENSG00000156299', 'ENSG00000156304', 'ENSG00000157538', 'ENSG00000157540', 'ENSG00000157542', 'ENSG00000157551', 'ENSG00000157554', 'ENSG00000157557', 'ENSG00000157578', 'ENSG00000157601', 'ENSG00000157617', 'ENSG00000159055', 'ENSG00000159079', 'ENSG00000159082', 'ENSG00000159086', 'ENSG00000159110',

In [36]:
chunk_idx=2
idx_start = (int(chunk_idx)-1) * int(genes_per_chunk)
idx_end = (int(chunk_idx)) * int(genes_per_chunk)
subset_gene = genes[idx_start:idx_end]
print("gene subset")
print(subset_gene)

gene subset
['ENSG00000215424', 'ENSG00000215454', 'ENSG00000215455', 'ENSG00000215458', 'ENSG00000221837', 'ENSG00000221859', 'ENSG00000221864', 'ENSG00000223262', 'ENSG00000223563', 'ENSG00000223662', 'ENSG00000223692', 'ENSG00000223768', 'ENSG00000223799', 'ENSG00000223901', 'ENSG00000224100', 'ENSG00000224269', 'ENSG00000224388', 'ENSG00000224421', 'ENSG00000224541', 'ENSG00000224574', 'ENSG00000224790', 'ENSG00000224905', 'ENSG00000225043', 'ENSG00000225330', 'ENSG00000225555', 'ENSG00000225731', 'ENSG00000226054', 'ENSG00000226501', 'ENSG00000227054', 'ENSG00000227256', 'ENSG00000227342', 'ENSG00000227406', 'ENSG00000227698', 'ENSG00000227757', 'ENSG00000228107', 'ENSG00000228137', 'ENSG00000228149', 'ENSG00000228318', 'ENSG00000228404', 'ENSG00000228677', 'ENSG00000229007', 'ENSG00000229046', 'ENSG00000229047', 'ENSG00000229356', 'ENSG00000229623', 'ENSG00000229880', 'ENSG00000230061', 'ENSG00000230212', 'ENSG00000230366', 'ENSG00000230479', 'ENSG00000230982', 'ENSG00000231068',

In [4]:
trio_path = "data/phased/wes_union_calls/200k/shapeit5/parents_with_invariant_sites/ukb_wes_union_calls_200k_shapeit5_parents_chr21.txt"

In [5]:
vcf_path = "data/phased/wes_union_calls/200k/shapeit5/parents_with_invariant_sites/ukb_wes_union_calls_200k_shapeit5_parents_chr21_tmp.vcf.gz"

In [20]:
ht = hl.import_table(trio_path, no_header=False)

2022-12-09 13:18:48 Hail: INFO: Reading table without type imputation
  Loading field 'ID' as type str (not specified)
  Loading field 'CHR' as type str (not specified)
  Loading field 'POS' as type str (not specified)
  Loading field 'REF' as type str (not specified)
  Loading field 'ALT' as type str (not specified)
  Loading field 'MAF' as type str (not specified)
  Loading field 'AF' as type str (not specified)
  Loading field 'AC' as type str (not specified)
  Loading field 'AN' as type str (not specified)
  Loading field 'HWE' as type str (not specified)
  Loading field 'trio_id' as type str (not specified)
  Loading field 'index' as type str (not specified)
  Loading field 'switch' as type str (not specified)
  Loading field 'switches' as type str (not specified)


In [21]:
ht = ht.annotate(varid = hl.delimit([ht.CHR, ht.POS, ht.REF, ht.ALT],":"))

In [9]:
vcf = io.import_table(vcf_path, "vcf")

In [10]:
mt = vcf.checkpoint("mt_vcf_deleteme_checkpoint.mt")

2022-12-09 11:47:59 Hail: INFO: scanning VCF for sortedness...
2022-12-09 11:55:07 Hail: INFO: Coerced sorted VCF - no additional import work to do
2022-12-09 13:04:27 Hail: INFO: wrote matrix table with 137771 rows and 199282 columns in 8 partitions to mt_vcf_deleteme_checkpoint.mt


In [11]:
mt = mt.annotate_rows(MAC = variants.get_mac_expr(mt))

In [12]:
#mt.filter_rows(mt.MAC==1).count()

(53795, 199282)

In [22]:
ht = ht.key_by(**hl.parse_variant(ht.varid))

In [27]:
ht = ht.annotate(new_AC=mt.rows().index(ht.key).info.AC)

In [29]:
ht.export("agreement.txt")

2022-12-09 13:20:39 Hail: INFO: Ordering unsorted dataset with network shuffle1]
2022-12-09 13:20:43 Hail: INFO: merging 2 files totalling 20.4M...  (0 + 1) / 1]
2022-12-09 13:20:43 Hail: INFO: while writing:
    agreement.txt
  merge time: 36.531ms


In [19]:
hl.set_global_seed(1)
mt = hl.balding_nichols_model(1, 100, 1000, phased=True)
rows = mt.count()[0]

2022-11-21 15:34:20 Hail: INFO: balding_nichols_model: generating genotypes for 1 populations, 100 samples, and 1000 variants...


In [20]:
locus = [ "chr%s:%s" % (21, str(i+1)) for i in range(rows)]
ref = [ get_tid(1) for i in range(rows)]
alt = [ get_tid(1) for i in range(rows)]

# convert to HailTable
df = pd.DataFrame({'locus':locus, 'ref':ref, 'alt':alt})
ht = hl.Table.from_pandas(df)
ht = ht.add_index()
ht = ht.key_by('idx')

In [22]:
mt = mt.add_row_index()
mt = mt.annotate_rows(
    gene_id=ht[mt.row_idx].ref
)

In [25]:
aggr = ko.aggr_phase_count_by_expr(mt, mt.gene_id)

In [26]:
aggr.show()

2022-11-21 15:37:30 Hail: INFO: Coerced sorted dataset
2022-11-21 15:37:31 Hail: INFO: Coerced sorted dataset
2022-11-21 15:37:31 Hail: INFO: Coerced sorted dataset
2022-11-21 15:37:31 Hail: INFO: Coerced sorted dataset
2022-11-21 15:37:32 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-11-21 15:37:33 Hail: INFO: Ordering unsorted dataset with network shuffle


+---------+-------------+-------------+------------+--------------+
| gene_id | 0.phased.a1 | 0.phased.a2 | 0.phased.n | 0.unphased.n |
+---------+-------------+-------------+------------+--------------+
| str     |       int64 |       int64 |      int64 |        int64 |
+---------+-------------+-------------+------------+--------------+
| "a"     |           5 |           5 |         10 |            0 |
| "b"     |           6 |           8 |         14 |            0 |
| "c"     |           6 |           7 |         13 |            0 |
| "d"     |           8 |           7 |         15 |            0 |
| "e"     |           8 |           4 |         12 |            0 |
| "f"     |           8 |           7 |         15 |            0 |
| "g"     |           9 |           8 |         17 |            0 |
| "h"     |           7 |           8 |         15 |            0 |
| "i"     |           7 |           3 |         10 |            0 |
| "j"     |           7 |           9 |         16 |            0 |
+---------+-------------+-------------+------------+--------------+

+-------------+
| 0.hom_alt_n |
+-------------+
|       int64 |
+-------------+
|          16 |
|          12 |
|          14 |
|          12 |
|           7 |
|          14 |
|           8 |
|          14 |
|          13 |
|          17 |
+-------------+
showing top 10 rows
showing the first 1 of 100 columns

In [35]:
p_old = "data/mt/old/annotated/ukb_eur_wes_union_calls_200k_chr21.mt/"
p_new = "data/mt/annotated/ukb_wes_union_calls_200k_chr21.mt/"
p_filter = "data/mt/prefilter/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp95.maf0_005.mt/"

In [36]:
mt1 = io.import_table(p_old, "mt", calc_info = False)
mt2 = io.import_table(p_new, "mt", calc_info = False)
mt3 = io.import_table(p_filter, "mt", calc_info = False)


In [12]:
keep = ["chr21:39199148:A:AG","chr21:39199143:TC:T"]

In [13]:
mt1 = mt1.filter_rows(hl.literal(keep).contains(mt1.varid))
mt2 = mt2.filter_rows(hl.literal(keep).contains(mt2.varid))
mt1 = mt1.checkpoint("remove_me_mt1_checkpoint.mt")
mt2 = mt2.checkpoint("remove_me_mt2_checkpoint.mt")

2022-11-21 16:56:19 Hail: INFO: wrote matrix table with 2 rows and 176915 columns in 508 partitions to remove_me_mt1_checkpoint.mt
2022-11-21 16:59:22 Hail: INFO: wrote matrix table with 2 rows and 176587 columns in 7 partitions to remove_me_mt2_checkpoint.mt


In [37]:
mt3 = mt3.filter_rows(hl.literal(keep).contains(mt3.varid))
mt3 = mt3.checkpoint("remove_me_mt3_checkpoint.mt")

2022-11-21 17:10:28 Hail: INFO: wrote matrix table with 2 rows and 176587 columns in 7 partitions to remove_me_mt3_checkpoint.mt


In [43]:
mt1 = hl.read_matrix_table("remove_me_mt1_checkpoint.mt/")
mt2 = hl.read_matrix_table("remove_me_mt2_checkpoint.mt/")

In [44]:
print(mt1.count()) 
print(mt2.count())

(2, 176915)
(2, 176587)


In [29]:
mt1.aggregate_entries(hl.agg.sum(mt1.GT.is_het()))

110

In [26]:
mt2.aggregate_entries(hl.agg.sum(mt2.GT.is_het()))

108

In [38]:
mt3.aggregate_entries(hl.agg.sum(mt3.GT.is_het()))

13

In [31]:
ko.aggr_count_calls(mt1)

(62, 48)

In [32]:
ko.aggr_count_calls(mt2)

(56, 52)

In [46]:
mt = mt2
pp_cutoff = 0.45
pp_cutoff = float(pp_cutoff)
expr_pp_cutoff = (mt.PP >= pp_cutoff) & (hl.is_defined(mt.GT))
expr_keep = ~(hl.is_defined(mt.PP)) & (hl.is_defined(mt.GT))
mt = mt.filter_entries((expr_pp_cutoff) | (expr_keep))
mt.aggregate_entries(hl.agg.sum(mt.GT.is_het()))

108

In [24]:
mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical)

NameError: name 'mt' is not defined

In [11]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
mt1 = mt1.annotate_rows(
    consequence_category=ko.csqs_case_builder(
            worst_csq_expr=mt1.consequence.vep.worst_csq_by_gene_canonical,
            use_loftee=True))
mt1 = mt1.filter_rows(hl.literal(set(csqs_category)).contains(mt1.consequence_category))
mt1.count()

Note: LOFTEE Low Confidence PTVs will be annotated as damaging_missense.


KeyboardInterrupt: 

In [69]:
mt1.aggregate_entries(hl.agg.sum(mt1.GT.is_het()))

110863

In [71]:
mt1.aggregate_entries(hl.agg.sum(hl.is_defined(mt1.GT)))

539451417

In [79]:
mt1.aggregate_entries(hl.agg.sum(ko.is_phased(mt1.GT)))

265195585

In [ ]:
mt1.varid

In [ ]:
mt1.filter_rows()

In [68]:
mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)
mt2 = mt2.annotate_rows(
    consequence_category=ko.csqs_case_builder(
            worst_csq_expr=mt2.consequence.vep.worst_csq_by_gene_canonical,
            use_loftee=True))
mt2 = mt2.filter_rows(hl.literal(set(csqs_category)).contains(mt2.consequence_category))
mt2.count()

Note: LOFTEE Low Confidence PTVs will be annotated as damaging_missense.


(3050, 176587)

In [70]:
mt2.aggregate_entries(hl.agg.sum(mt2.GT.is_het()))

110715

In [72]:
mt2.aggregate_entries(hl.agg.sum(hl.is_defined(mt2.GT)))

538590350

In [80]:
mt2.aggregate_entries(hl.agg.sum(ko.is_phased(mt2.GT)))

538590350

In [81]:
# mt1 defined: 539.451.417
# mt1 phased:  265.195.585
# mt2 defined: 538.590.350
# mt2 phased:  538.590.350

-273394765

In [4]:
p = "data/mt/prefilter/test/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp95.maf0_005.mt/"
mt = io.import_table(p, "mt", calc_info = True)

In [6]:
csqs_category = "pLoF"
x = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.collect()

In [32]:
mt = mt.filter_rows(hl.literal(set(csqs_category)).contains(mt.consequence_category))

In [33]:
mt.show()

KeyboardInterrupt: 

In [99]:
gene = ko.aggr_phase_count_by_expr(mt, mt.consequence.vep.worst_csq_by_gene_canonical.gene_id)

In [102]:
a1 = gene.phased.a1.collect()
a2 = gene.phased.a2.collect()

In [107]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.show()

KeyboardInterrupt: 

In [ ]:
genes = ko.aggr_phase_count_by_expr(mt, gene_expr)


In [47]:
#p = "data/mt/annotated/ukb_wes_union_calls_200k_chr21.mt/"
#p = "data/mt/prefilter/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp95.maf0_005.mt/"
#p = "data/knockouts/alt/test/ukb_eur_wes_200k_chr20_pLoF_checkpoint.mt/"

In [ ]:
        expr_pko = ko.calc_prob_ko(genes.hom_alt_n, genes.phased, genes.unphased, only_homs=False)
        expr_ko = ko.annotate_knockout(genes.hom_alt_n, expr_pko)

In [44]:
mt = io.import_table(p, "mt", calc_info = False)
mt = mt.explode_rows(mt.consequence.vep.worst_csq_by_gene_canonical)
mt = mt.annotate_rows(
    consequence_category=ko.csqs_case_builder(
            worst_csq_expr=mt.consequence.vep.worst_csq_by_gene_canonical,
            use_loftee=True))

Note: LOFTEE Low Confidence PTVs will be annotated as damaging_missense.


In [46]:
# subset to current csqs category
csqs_category = ["pLoF"] # "damaging_missense"
mt = mt.filter_rows(hl.literal(set(csqs_category)).contains(mt.consequence_category))
n_csqs = mt.count()[0]
sys.stderr.write(f"Filtering to {n_csqs} variants that are {csqs_category}.")

Filtering to 3050 variants that are ['pLoF'].==============>        (6 + 1) / 7]

45

In [48]:
mt = io.import_table(p, "mt", calc_info = False)

--------------------------------------------------------
Type:
        struct {
        rsid: str, 
        qual: float64, 
        filters: set<str>, 
        info: struct {
            AF: float64, 
            AQ: array<int32>, 
            AC: int32, 
            AN: int32, 
            ExcessHet: float64
        }, 
        vep: struct {
            assembly_name: str, 
            allele_string: str, 
            ancestral: str, 
            ensp: str, 
            minimised: int32, 
            colocated_variants: array<struct {
                af: float64, 
                afr_af: float64, 
                amr_af: float64, 
                eas_af: float64, 
                eur_af: float64, 
                sas_af: float64, 
                aa_af: float64, 
                ea_af: float64, 
                gnomAD_AF: float64, 
                gnomAD_AFR_AF: float64, 
                gnomAD_AMR_AF: float64, 
                gnomAD_ASJ_AF: float64, 
                gnomAD_EAS_AF: f

In [29]:
#mt = io.import_table(p, "mt", calc_info = False)
#mt.count()

(141807, 176587)

(540, 176587)

In [12]:
list(set(x))

['non_coding',
 'synonymous',
 'pLoF',
 'damaging_missense',
 'other_missense',
 None]

In [3]:
#p1 = "data/phased/wes_union_calls/200k/shapeit5/parents/ukb_wes_union_calls_200k_shapeit5_parents_chr20.vcf.gz"
#p2 = "data/phased/wes_union_calls/200k/shapeit4/with_parents/ukb_eur_wes_union_calls_200k_chr20.vcf.bgz"
p1 = 'data/unphased/wes/post-qc/ukb_wes_200k_filtered_chr20.mt/'
io.import_table(p1, "mt", calc_info = False).count()

(332423, 199795)

In [4]:
p2 = 'data/unphased/wes/prefilter/200k/ukb_prefilter_wes_200k_chr20.mt/'
io.import_table(p2, "mt", calc_info = False).count()

(310177, 199634)

In [5]:
p3 = 'data/unphased/wes/prefilter/200k/ukb_split_wes_200k_chr20_parents.vcf.bgz/'
io.import_table(p3, "vcf", calc_info = False).count()

2022-11-18 16:23:38 Hail: INFO: scanning VCF for sortedness...
2022-11-18 16:23:53 Hail: INFO: Coerced sorted VCF - no additional import work to do


(8200, 352)

In [6]:
p4 = "data/unphased/wes_union_calls/old/without_trio_parents/ukb_wes_union_calls_200k_chr20_parents.vcf.bgz"
mt = io.import_table(p4, "vcf", calc_info = True)
mt.count()

2022-11-18 16:23:54 Hail: INFO: scanning VCF for sortedness...
2022-11-18 16:23:55 Hail: INFO: Coerced sorted VCF - no additional import work to do


(194635, 352)

In [7]:
mt.filter_rows(hl.min(mt.info.AC) >= 1).count()

(27145, 352)

In [8]:
p5 = "data/unphased/wes_union_calls/prefilter_no_maf_cutoff/200k/ukb_wes_union_calls_chr20_parents.vcf.gz"
mt = io.import_table(p5, "vcf", calc_info = True)
mt.count()

2022-11-18 16:24:05 Hail: INFO: scanning VCF for sortedness...
2022-11-18 16:24:06 Hail: INFO: Coerced sorted VCF - no additional import work to do


(24744, 352)

In [9]:
mt.filter_rows(hl.min(mt.info.AC) >= 1).count()

(24744, 352)

In [14]:
p6 = "data/unphased/calls/prefilter/200k/ukb_split_calls_200k_chr20_no_parents.mt/"
mt = io.import_table(p6, "mt", calc_info = True)
mt.count()

(17628, 199282)

In [15]:
p7 = "data/unphased/calls/prefilter_no_maf_cutoff/200k/ukb_split_calls_200k_chr20_no_parents.mt/"
mt = io.import_table(p7, "mt", calc_info = True)
mt.count()

(18954, 199282)

In [17]:
p8 = "data/unphased/calls/liftover/ukb_liftover_calls_500k_chr20.mt/"
mt = io.import_table(p8, "mt", calc_info = True)
mt.count()

(19958, 488377)

In [26]:
p9 = "data/unphased/wes_union_calls/200k_from_500k/ukb_wes_union_calls_chr20_parents.vcf.gz"
mt = io.import_table(p9, "vcf", calc_info = True)
mt.count()

2022-11-18 16:32:02 Hail: INFO: scanning VCF for sortedness...
2022-11-18 16:32:03 Hail: INFO: Coerced sorted VCF - no additional import work to do


(24324, 352)

In [10]:
#p6 = "data/phased/wes_union_calls/200k/shapeit5/ligated/ukb_wes_union_calls_200k_chr20.vcf.bgz"
#mt = io.import_table(p6, "vcf", calc_info = True)
#mt.count()

In [14]:
#mt1 = io.import_table(p1, "vcf")
#mt2 = io.import_table(p2, "vcf")

In [11]:
#print(mt1.count())
#print(mt2.count())

In [83]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/calls/shapeit5/200k_from_500k/old/test2_ukb_phased_calls_200k_from_500k_chr21.vcf.bgz"

In [84]:
mt = io.import_table(path, "vcf", calc_info = False)

2022-11-21 14:47:46 Hail: WARN: '/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/calls/shapeit5/200k_from_500k/old/test2_ukb_phased_calls_200k_from_500k_chr21.vcf.bgz' refers to no files


FatalError: HailException: arguments refer to no files

Java stack trace:
is.hail.utils.HailException: arguments refer to no files
	at is.hail.utils.ErrorHandling.fatal(ErrorHandling.scala:17)
	at is.hail.utils.ErrorHandling.fatal$(ErrorHandling.scala:17)
	at is.hail.utils.package$.fatal(package.scala:78)
	at is.hail.io.vcf.LoadVCF$.globAllVCFs(LoadVCF.scala:1151)
	at is.hail.io.vcf.MatrixVCFReader$.apply(LoadVCF.scala:1590)
	at is.hail.io.vcf.MatrixVCFReader$.fromJValue(LoadVCF.scala:1665)
	at is.hail.expr.ir.MatrixReader$.fromJson(MatrixIR.scala:89)
	at is.hail.expr.ir.IRParser$.matrix_ir_1(Parser.scala:1745)
	at is.hail.expr.ir.IRParser$.$anonfun$matrix_ir$1(Parser.scala:1665)
	at is.hail.utils.StackSafe$More.advance(StackSafe.scala:64)
	at is.hail.utils.StackSafe$.run(StackSafe.scala:16)
	at is.hail.utils.StackSafe$StackFrame.run(StackSafe.scala:32)
	at is.hail.expr.ir.IRParser$.$anonfun$parse_matrix_ir$1(Parser.scala:2025)
	at is.hail.expr.ir.IRParser$.parse(Parser.scala:2010)
	at

In [130]:
mt = io.recalc_info(mt)
mt = mt.annotate_rows(AC=mt.info.AC)
mt = mt.transmute_rows(AC = mt.info.annotate(AC = mt.info.AC))

In [132]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'AC': struct {
        AC: int32, 
        AN: int32, 
        AF: float64
    }
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


TypeError: drop: parameter '*fields' (arg 0 of 1): expected str, found hail.expr.expressions.typed_expressions.ArrayNumericExpression: <ArrayNumericExpression of type array<float64>>

In [81]:
mt = mt.annotate_rows(
    info = mt.info.annotate(
        AN = hl.agg.call_stats(mt[gt_field], mt.alleles).AN
    )
)

AttributeError: MatrixTable instance has no field, method, or property 'info'
    Hint: use 'describe()' to show the names of all data fields.

In [79]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'AN': int32
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [67]:
mt.transmute_rows(info = mt.info.annotate(AC = mt.info.AC[1])).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AC: int32, 
        AN: int32, 
        AF: array<float64>
    }
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [51]:
mt.info.AC.show()

2022-11-03 16:06:45 Hail: INFO: scanning VCF for sortedness...
2022-11-03 16:07:13 Hail: INFO: Coerced sorted VCF - no additional import work to do


,,
locus,alleles,
locus<GRCh38>,array<str>,array<int32>
chr21:10414352,"[""A"",""G""]","[13605,384959]"
chr21:10510532,"[""T"",""C""]","[379631,18933]"
chr21:10514459,"[""A"",""G""]","[6893,391671]"
chr21:10544690,"[""C"",""T""]","[10294,388270]"
chr21:10550046,"[""G"",""C""]","[18953,379611]"
chr21:10567789,"[""G"",""A""]","[2172,396392]"
chr21:10570491,"[""T"",""C""]","[1880,396684]"
chr21:10638865,"[""C"",""T""]","[387079,11485]"


In [24]:
def recalc_info(mt, maf=None, info_field='info', gt_field='GT'):
    r'''Recalculate INFO fields AF, AC, AN and keep sites with minor allele
    frequency > `maf The fields AF and AC are made into integers instead of
    an array, only showing the AF and AC for the alt allele
    '''
    if info_field in mt.row:
        mt = mt.annotate_rows(
            info=mt[info_field].annotate(**hl.agg.call_stats(mt[gt_field][1],
                                                             mt.alleles)
                                         ).drop('homozygote_count')
        )  # recalculate AF, AC, AN, drop homozygote_count
    else:
        mt = mt.annotate_rows(
            **{info_field: hl.agg.call_stats(mt[gt_field], mt.alleles)})
    mt = mt.annotate_rows(
        **{info_field: mt[info_field].annotate(
            **{field: mt[info_field][field][1] for field in ['AF', 'AC']}
        )}
    )  # keep only the 2nd entries in AF and AC, which correspond to the alt alleles
    if maf is not None:
        mt = mt.filter_rows(
            (mt[info_field].AF > maf) & (
                mt[info_field].AF < (
                    1 - maf)))
    return mt

In [7]:
mt.info.AC.show()

2022-11-03 15:56:30 Hail: INFO: scanning VCF for sortedness...
2022-11-03 15:57:03 Hail: INFO: Coerced sorted VCF - no additional import work to do


,,
locus,alleles,
locus<GRCh38>,array<str>,array<int32>
chr21:10414352,"[""A"",""G""]",[384959]
chr21:10510532,"[""T"",""C""]",[18933]
chr21:10514459,"[""A"",""G""]",[391671]
chr21:10544690,"[""C"",""T""]",[388270]
chr21:10550046,"[""G"",""C""]",[379611]
chr21:10567789,"[""G"",""A""]",[396392]
chr21:10570491,"[""T"",""C""]",[396684]
chr21:10638865,"[""C"",""T""]",[11485]


In [5]:
mt = io.import_table(path, "vcf", calc_info = False)

In [8]:
samples = mt.s.collect()

2022-10-18 11:48:22 Hail: INFO: scanning VCF for sortedness...
2022-10-18 11:52:16 Hail: INFO: Coerced sorted VCF - no additional import work to do


In [11]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [45]:
def read_interval_idx(interval_path, idx):
    """ Read a specific interval index from a file
    :param interval_path: path to interval file (assumine one line interval)
    :param idx: interval index
    """
    assert idx >= 0
    with open(interval_path, "r") as infile:
        for i, line in enumerate(infile):
            if i == idx:
                return(line)
    


In [39]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/wes_union_calls/prephased/chunks/intervals/intervals_spc1000_chr22.tsv"

In [49]:
path = "/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/phased/wes_union_calls/prephased/ukb_eur_wes_union_calls_prephased_200k_chr22.vcf.bgz.tmp"

In [62]:
lines = []
with open(path, "r") as infile:
    for line in infile:
        if line.strip():
            lines += [line.strip()]


In [63]:
lines

['data/phased/wes_union_calls/prephased/chunks/ukb_eur_wes_union_calls_200k_chr22-short/chr22_spc100.1of1994.phased.vcf.gz',
 'data/phased/wes_union_calls/prephased/chunks/ukb_eur_wes_union_calls_200k_chr22-short/chr22_spc100.2of1994.phased.vcf.gz']

In [4]:
fam = samples.get_fam()

2022-09-30 14:10:15 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)


In [ ]:
fam.

In [65]:
def exclude_trio_parents(mt):
    '''Filter to children of trio parents in the data
    '''
    import hail as hl

    # filter fam file by input samples
    fam = samples.get_fam()
    fam = fam.filter(hl.is_defined(mt.cols()[fam.IID]))
    fam = fam.filter(hl.literal(["TRIO"]).contains(fam.REL))
    
    # get ID of parents
    pids = []
    pids.extend(fam.PAT.collect())
    pids.extend(fam.MAT.collect())
    pids = [x for x in pids if x != "0"]
    assert len(pids) % 2 == 0
    
    # remove by paternal IDs
    return mt.filter_cols(~hl.literal(pids).contains(mt.s))


In [50]:
input_path = "data/mt/annotated/ukb_eur_wes_union_calls_200k_chr21.mt"

In [51]:
mt = io.import_table(input_path, "mt", calc_info = False)

In [52]:
mt.count()

(99869, 176915)

In [66]:
x = exclude_trio_parents(mt)

2022-09-30 14:46:56 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)


479203


2022-09-30 14:46:57 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:46:58 Hail: INFO: Coerced sorted dataset


173671


2022-09-30 14:47:01 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:02 Hail: INFO: Coerced sorted dataset


399


2022-09-30 14:47:05 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:06 Hail: INFO: Coerced sorted dataset
2022-09-30 14:47:10 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:47:10 Hail: INFO: Coerced sorted dataset


In [85]:
def exclude_parents_by_fam(mt, relation = ["TRIO","DUO"]):
    '''

    : param mt: MatrixTable containing indiviudals to filter
    : param relation: list containing relationships to filter by, for example "TRIO" filters
    excludes parents in trios, whereas ["TRIO", "DUO"] excludes parents in trios or duo pedigrees.
    '''

    import hail as hl

    assert len(relation) > 0
    # filter fam file by input samples
    fam = samples.get_fam()
    fam = fam.filter(hl.is_defined(mt.cols()[fam.IID]))
    fam = fam.filter(hl.literal(relation).contains(fam.REL))

    # get ID of parents
    pids = []
    pids.extend(fam.PAT.collect())
    pids.extend(fam.MAT.collect())
    pids = [x for x in pids if x != "0"]
    assert len(pids) % 2 == 0
    
    # remove by paternal IDs
    mt = mt.filter_cols(~hl.literal(pids).contains(mt.s))
    return mt

In [86]:
x = filter_to_offspring_by_fam(mt, relation = ["TRIO","DUO"])

2022-09-30 14:56:34 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)
2022-09-30 14:56:35 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:56:35 Hail: INFO: Coerced sorted dataset
2022-09-30 14:56:39 Hail: INFO: Ordering unsorted dataset with network shuffle
2022-09-30 14:56:39 Hail: INFO: Coerced sorted dataset


In [87]:
mt.count()

(99869, 176915)

In [88]:
x.count()

(99869, 176012)

In [2]:
import pysam

ModuleNotFoundError: No module named 'pysam'